In [2]:
# ==========================================================
# E8: Classical Logistic Regression Baseline
# Dataset: MNIST (0 vs 1)
# PCA Components: 4
# Train Samples: 2000
# Test Samples: 500
# ==========================================================

import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

# ==========================================================
# STEP 1: Load MNIST
# ==========================================================

print("Loading MNIST...")

mnist = fetch_openml('mnist_784', version=1, as_frame=False)

X = mnist.data.astype(np.float32)
y = mnist.target.astype(int)

# ==========================================================
# STEP 2: Select Digits 0 and 1
# ==========================================================

mask = (y == 0) | (y == 1)

X = X[mask]
y = y[mask]

print("Total samples:", len(y))

# ==========================================================
# STEP 3: Stratified Split
# ==========================================================

X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# ==========================================================
# STEP 4: Select Exactly 2000 Train and 500 Test Samples
# ==========================================================

np.random.seed(42)

train_idx = np.random.choice(
    len(X_train_full),
    2000,
    replace=False
)

test_idx = np.random.choice(
    len(X_test_full),
    500,
    replace=False
)

X_train = X_train_full[train_idx]
y_train = y_train_full[train_idx]

X_test = X_test_full[test_idx]
y_test = y_test_full[test_idx]

print("\nTraining Samples:", len(X_train))
print("Testing Samples :", len(X_test))

# ==========================================================
# STEP 5: PCA (4 Components)
# ==========================================================

pca = PCA(n_components=4)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("\nExplained Variance Ratio")

for i, var in enumerate(pca.explained_variance_ratio_):
    print(f"PC{i+1}: {var*100:.2f}%")

print(
    f"\nTotal Variance Retained: "
    f"{sum(pca.explained_variance_ratio_)*100:.2f}%"
)

# ==========================================================
# STEP 6: Logistic Regression
# ==========================================================

model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

model.fit(
    X_train_pca,
    y_train
)

# ==========================================================
# STEP 7: Prediction
# ==========================================================

y_pred = model.predict(X_test_pca)

# ==========================================================
# STEP 8: Evaluation
# ==========================================================

acc = accuracy_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)

print("\n==========================")
print("E8 RESULTS")
print("==========================")

print(f"Accuracy = {acc*100:.2f}%")
print(f"F1 Score = {f1:.4f}")

print("\nClassification Report")
print(classification_report(
    y_test,
    y_pred
))

Loading MNIST...
Total samples: 14780

Training Samples: 2000
Testing Samples : 500

Explained Variance Ratio
PC1: 32.17%
PC2: 8.91%
PC3: 8.28%
PC4: 5.69%

Total Variance Retained: 55.05%

E8 RESULTS
Accuracy = 99.60%
F1 Score = 0.9962

Classification Report
              precision    recall  f1-score   support

           0       0.99      1.00      1.00       234
           1       1.00      0.99      1.00       266

    accuracy                           1.00       500
   macro avg       1.00      1.00      1.00       500
weighted avg       1.00      1.00      1.00       500

